# Разведочный анализ данных

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:.4f}".format)

TARGET = "Y house price of unit area"


def find_project_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd / "1_linear_regression_house_prices"]
    candidates.extend(cwd.parents)
    candidates.extend(parent / "1_linear_regression_house_prices" for parent in cwd.parents)

    seen = set()
    for candidate in candidates:
        try:
            path = candidate.resolve()
        except OSError:
            continue
        if path in seen:
            continue
        seen.add(path)
        if (path / "data" / "prices_train.csv").exists() and (path / "data" / "prices_test.csv").exists():
            return path
    raise FileNotFoundError("Не найдена папка проекта с data/prices_train.csv")


PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / "data"
TRAIN_PATH = DATA_DIR / "prices_train.csv"
TEST_PATH = DATA_DIR / "prices_test.csv"

train = pd.read_csv(TRAIN_PATH, index_col=0)
test = pd.read_csv(TEST_PATH, index_col=0)
feature_cols = [column for column in train.columns if column != TARGET]

PROJECT_DIR

## 1. Структура данных


In [ ]:
overview = pd.DataFrame(
    {
        "rows": [len(train), len(test)],
        "columns": [train.shape[1], test.shape[1]],
    },
    index=["train", "test"],
)

column_info = pd.DataFrame({"column": train.columns})
column_info = column_info.assign(
    train_dtype=column_info["column"].map(lambda column: str(train[column].dtype) if column in train else None),
    test_dtype=column_info["column"].map(lambda column: str(test[column].dtype) if column in test else None),
)
column_info["test_dtype"] = column_info["test_dtype"].fillna("NaN")

train_preview = train[feature_cols + [TARGET]].head().rename_axis("index")
test_preview = test[feature_cols].head().rename_axis("index")

display(overview)
display(column_info)
display(train_preview)
display(test_preview)

## 2. Пропуски

In [ ]:
missing = pd.DataFrame(index=train.columns.union(test.columns))
missing["train_missing"] = train.isna().sum().reindex(missing.index)
missing["train_missing_pct"] = missing["train_missing"] / len(train) * 100
missing["test_missing"] = test.isna().sum().reindex(missing.index)
missing["test_missing_pct"] = missing["test_missing"] / len(test) * 100
missing = missing.fillna(0).sort_values(["train_missing", "test_missing"], ascending=False)

duplicates = pd.DataFrame(
    {
        "duplicated_rows": [train.duplicated().sum(), test.duplicated().sum()],
    },
    index=["train", "test"],
)

display(missing)
display(duplicates)

## 3. Описательные статистики

In [ ]:
display(train.describe().T)
display(test.describe().T)

## 4. Выбросы

Критерий: `x < Q1 - 1.5 * IQR` или `x > Q3 + 1.5 * IQR`

In [ ]:
def iqr_outlier_table(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    rows = []
    for column in columns:
        values = df[column].dropna()
        q1, q3 = values.quantile([0.25, 0.75])
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        mask = (values < lower) | (values > upper)
        rows.append(
            {
                "column": column,
                "outliers": int(mask.sum()),
                "outliers_pct": mask.sum() / len(df) * 100,
                "lower_bound": lower,
                "upper_bound": upper,
                "min": values.min(),
                "max": values.max(),
            }
        )
    return pd.DataFrame(rows).sort_values("outliers", ascending=False)


outliers_train = iqr_outlier_table(train, list(train.columns))
outliers_test = iqr_outlier_table(test, list(test.columns))

display(outliers_train)
display(outliers_test)

plot_columns = feature_cols + [TARGET]
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.ravel()
for ax, column in zip(axes, plot_columns):
    sns.boxplot(y=train[column], ax=ax, color="#5B8DEF")
    ax.set_title(column, fontsize=10)
    ax.set_xlabel("")
for ax in axes[len(plot_columns):]:
    ax.axis("off")
plt.tight_layout()

## 5. Целевая переменная

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(train[TARGET], kde=True, bins=30, ax=axes[0], color="#2F7D6E")
sns.histplot(np.log1p(train[TARGET]), kde=True, bins=30, ax=axes[1], color="#9A5B00")
axes[0].set_title("Исходная целевая переменная")
axes[1].set_title("log1p целевой переменной")
plt.tight_layout()

## 6. Связи признаков с ценой

In [ ]:
raw_relationships = [
    "X1 transaction date",
    "X2 house age",
    "X3 distance to the nearest MRT station",
    "X4 number of convenience stores",
    "X5 latitude",
    "X6 longitude",
]

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for ax, column in zip(axes.ravel(), raw_relationships):
    data = train[[column, TARGET]].dropna()
    sns.regplot(data=data, x=column, y=TARGET, ax=ax, scatter_kws={"alpha": 0.55, "s": 28}, line_kws={"color": "#B03A2E"})
    ax.set_title(column, fontsize=10)
plt.tight_layout()

## 7. Корреляции признаков

In [ ]:
raw_corr = train[feature_cols + [TARGET]].corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(raw_corr, annot=True, fmt=".2f", cmap="vlag", center=0, square=True, linewidths=0.5)
plt.title("Корреляционная матрица")
plt.tight_layout()

## 8. Географическая структура

In [ ]:
geo_train = train[["X5 latitude", "X6 longitude", TARGET]].dropna()
geo_test = test[["X5 latitude", "X6 longitude"]].dropna()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
points = axes[0].scatter(
    geo_train["X6 longitude"],
    geo_train["X5 latitude"],
    c=geo_train[TARGET],
    cmap="viridis",
    s=45,
    alpha=0.85,
)
axes[0].set_title("Координаты train и цена")
axes[0].set_xlabel("X6 longitude")
axes[0].set_ylabel("X5 latitude")
fig.colorbar(points, ax=axes[0], label=TARGET)

axes[1].scatter(
    geo_train["X6 longitude"],
    geo_train["X5 latitude"],
    s=35,
    alpha=0.65,
    label="train",
)
axes[1].scatter(
    geo_test["X6 longitude"],
    geo_test["X5 latitude"],
    s=35,
    alpha=0.65,
    label="test",
)
axes[1].set_title("Пространственное покрытие train и test")
axes[1].set_xlabel("X6 longitude")
axes[1].set_ylabel("X5 latitude")
axes[1].legend()
plt.tight_layout()

## Вывод

1. Пропуски есть в возрасте дома, расстоянии до MRT и числе магазинов.
2. Описательная статистика показывает различный масштаб признаков, особенно для расстояния до MRT; признаки потребуется масштабировать.
3. Выбросы по IQR наиболее заметны в расстоянии до MRT, координатах и цене.
4. Целевая переменная имеет правый хвост: основная масса объектов находится в среднем ценовом диапазоне, но есть отдельные дорогие наблюдения.
5. Наиболее выраженные визуальные связи с ценой дают расстояние до MRT, число магазинов рядом и координаты объекта.
6. Корреляционная матрица подтверждает ключевую роль расстояния до MRT и географических признаков.
7. Train и test расположены в близком диапазоне координат, тестовая выборка не выглядит как отдельная географическая область.